In [1]:
import os
os.environ["PYTHONHASHSEED"] = "42"

import random
random.seed(42)

import numpy as np
np.random.seed(42)

def cutmax(
    X: np.ndarray,
    p: int = 9,
    T: int = 6,
    c: float = 1.0,
    eps: float = 1e-12,
    print_topk: int | None = 10,
    return_history: bool = False
):
    def _print_topk(label: str, V: np.ndarray, V_: np.ndarray, X0: np.ndarray, k: int):
        k = min(k, V.size)
        idx_k = np.argpartition(V, -k)[-k:]
        idx_sorted = idx_k[np.argsort(-V[idx_k])]
        print(f"[{label}] Top {k} elements:")
        if label == "Final Z (normalized)":
            for rank, i in enumerate(idx_sorted, 1):
                print(f"  {rank:2d}) idx={i:6d}, Y={V[i]:.6g}, Z={V_[i]:.6g}, X={X0[i]:.6g}")
        else:
            for rank, i in enumerate(idx_sorted, 1):
                print(f"  {rank:2d}) idx={i:6d}, u={V[i]:.6g}, u^p={V_[i]:.6g}, X={X0[i]:.6g}")
        print("-" * 60)

    X = np.asarray(X, dtype=np.float64).reshape(-1)
    Y = X.copy()
    hist = [Y.copy()] if return_history else None

    for t in range(1, T + 1):
        mu = Y.mean()
        Y_ = Y - mu
        pos_cnt = np.count_nonzero(Y_ > 0)
        nonpos_cnt = np.count_nonzero(Y_ <= 0)
        print(f"count(Y_ > 0) = {pos_cnt} count(Y_ <= 0) = {nonpos_cnt}")
        s2 = np.mean((Y_) ** 2)
        print(f'mean: {mu}')
        print(f'c   : {c}')
        print(f'std : {np.sqrt(s2)}')
        u = (Y_)/(c * np.sqrt(s2 + eps)) + 1.0
        Y = u ** p

        if return_history:
            hist.append(Y.copy())

        if isinstance(print_topk, int) and print_topk > 0:
            _print_topk(f"Iter {t}", u, Y, X, print_topk)

    S = Y.sum()
    Z = np.zeros_like(Y)
    if np.abs(S) >= eps:
        Z = Y / S

    if isinstance(print_topk, int) and print_topk > 0:
        _print_topk("Final Z (normalized)", Y, Z, X, print_topk)

    return (Z, hist) if return_history else Z


# z = np.random.uniform(-10, 10, 50257).astype(np.float64)
z = np.random.normal(loc=0.0, scale=5.0, size=50257).astype(np.float64)


p, T, c = 15, 3, 5.0
Z = cutmax(z, p=p, T=T, c=c, print_topk=10, return_history=False)

approx_argmax = int(np.argmax(Z))
true_argmax   = int(np.argmax(z))
print(f"approx argmax = {approx_argmax}, true argmax = {true_argmax}")


count(Y_ > 0) = 25163 count(Y_ <= 0) = 25094
mean: -0.002020634389272638
c   : 5.0
std : 5.002230918597458
[Iter 1] Top 10 elements:
   1) idx= 15843, u=1.8955, u^p=14650.4, X=22.3954
   2) idx= 18851, u=1.7882, u^p=6112.58, X=19.7117
   3) idx=  2895, u=1.78498, u^p=5949.68, X=19.6312
   4) idx=   209, u=1.77028, u^p=5255.84, X=19.2637
   5) idx= 33320, u=1.74531, u^p=4247.24, X=18.6392
   6) idx= 28552, u=1.73808, u^p=3990.55, X=18.4581
   7) idx= 21620, u=1.72203, u^p=3472.21, X=18.0568
   8) idx= 20126, u=1.72033, u^p=3421.06, X=18.0142
   9) idx= 32820, u=1.72024, u^p=3418.57, X=18.0121
  10) idx= 41670, u=1.70707, u^p=3046.33, X=17.6827
------------------------------------------------------------
count(Y_ > 0) = 7193 count(Y_ <= 0) = 43064
mean: 18.3168216591033
c   : 5.0
std : 119.84109181286016
[Iter 2] Top 10 elements:
   1) idx= 15843, u=25.4192, u^p=1.19516e+21, X=22.3954
   2) idx= 18851, u=11.1706, u^p=5.26175e+15, X=19.7117
   3) idx=  2895, u=10.8987, u^p=3.63602e+15, X=